# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [1]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }
    # TODO: loop up to `tries` times
    for attempt in range(tries): #creating a loop for tries, the tries variable is already defined in the def of function statement

    # TODO:   request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
      r = requests.get(BASE, params = params, timeout=10) #set a timeout so that if the server doesn't answer it doesn't hang our notebook

      if r.status_code == 200: #if the status code is 200 then it will return the data frame we want, daily
        return pd.DataFrame(r.json()['daily']) #returning dataframe from the daily block
    # TODO:   raise on a status that is not in RETRY_STATUSES
      if r.status_code not in RETRY_STATUSES:
        raise RuntimeError(f'{BASE} returned {r.status_code}: {r.text[:200]}') #if the satus code isn't in the included statuses, so a 400 or 404, it will raise a RuntimeError so we know where to look to fix the code
    # TODO:   otherwise sleep (increasing each time) and try again
      if attempt < tries - 1: #we want to stop after a certain amount of tries, so attempts has to be less than tries for us to keep running the code
        time.sleep(2 ** attempt) #makes the program wait before trying the code again since some code will succeed on later runs
    # TODO: raise if every attempt failed
    raise RuntimeError(f'{BASE} failed after {tries} attempts with {r.status_code} status code: {r.text[:200]}') #if each try failed, a RuntimeError will be displayed to let you know it failed
    pass

**Markdown:** Here we finished building the get_weather function so that it would raise errors when the function was failing. This is important because then we know we must look back at our code to fix something that has gone wrong. Additionally, we made sure to add tries and a timeout so that our function doesn't keep running when it is not supposed to which could potentially limit our access to the API if it needs delayed access attempts to run.

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [2]:
# TODO: call get_weather once for a range that spans both dates,
wx = get_weather('2026-09-05','2026-10-03') #calls the get weather function for a range starting with 2026-09-05 and going to 2026-10-03
#       then keep just the two dates you care about
wx = wx[wx['time'].isin(['2026-09-05','2026-10-03'])] #this looks inside our weather range and only keeps the data if the time includes the given dates we want
used_fallback = False #don't have to use fallback since we're pulling directly from the API
print(wx)

          time  temperature_2m_max  precipitation_sum
0   2026-09-05                29.4                2.5
28  2026-10-03                22.0               14.7


**Markdown:** Here we pulled weather from the API but only limited it to two dates instead of a whole range. This is important because sometimes you want to compare specific dates and not a whole range because there could be some days that are more important than others or more relevant. By selecting certain dates we can also spend more time analyzing as opposed to too much data which could be overwhelming to look at.

### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [3]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

**Markdown:** Since my Part 2 code ran fine, I did not use the fallback data and therefore did not have to complete Part 3. The information for adding a source column is down in Part 4.

### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [4]:
# TODO
wx = wx.rename(columns = {'time': 'date', 'temperature_2m_max':'temp_max','precipitation_sum':'precip_mm'}) #renaming all the columns from the old names to the new ones
wx['source'] = 'open-meteo' #adding to the dataframe a new column called source, since we pulled directly from open-meteo that is the value for source
wx['date'] = pd.to_datetime(wx['date']) #this makes the column a real datetime value
wx = wx[['date','temp_max','precip_mm','source']] #this ensures that we are including only the columns we want and in the order we want
print(wx)

         date  temp_max  precip_mm      source
0  2026-09-05      29.4        2.5  open-meteo
28 2026-10-03      22.0       14.7  open-meteo


**Markdown:** Here we renamed the columns to help us better understand the data and added a new column so that we can go back to the data as see where it came from. It is important to change the names as to provide more clarification as to what each piece of data means because the API might have more abstract or confusing names. It is also important to add a source column so that you can confirm the data later and ensure it is valid.

### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [5]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


**Markdown:** We validated the data because it makes sure that we are doing things correctly before we start doing analysis or pulling data from an errored data frame. It is important to make assertions so that you know your code is doing what you want it to do.

### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [6]:
# TODO
wettest = wx.loc[wx['precip_mm'].idxmax()] #finding the max precipiation value in the data frame since this will be the wetter date
driest = wx.loc[wx['precip_mm'].idxmin()] #finding the miniminum precipiation value in the data frame since this will be the dryer date
difference = wettest['precip_mm'] - driest['precip_mm'] #subtracting wet - dry to see how big the difference in precipiation was
print('The wettest day was', wettest['date'].date(), 'with', wettest['precip_mm'], 'mm.')
print('This was', difference, 'mm more than on', driest['date'].date(),'.')

The wettest day was 2026-10-03 with 14.7 mm.
This was 12.2 mm more than on 2026-09-05 .


**Markdown:** Here we compared precipitation for two dates to see which one had more precipitation, and therefore was wetter. We did this because it is not just enough to gather a lot of data, the whole point is to analyze it and see how it could be relevant to what you are doing or how you could use it.

### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [7]:
# TODO
from datetime import datetime
print('Endpoint:', BASE) #tell you if it came from the archive endpoint or forecast endpoint
print('Coordinates: lat=38.03, lon=-78.51') #given in get weather original function
print('Date range: 2026-09-05 to 2026-10-03') #these are the dates we are told to look at
print('Pulled: ', datetime.now().strftime('%Y-%m-%d %H:%M')) #tells what time the functions to pull the data was ran
print('Source: Real data from Open-Meteo historical archive API') #we pull from the real data


Endpoint: https://archive-api.open-meteo.com/v1/archive
Coordinates: lat=38.03, lon=-78.51
Date range: 2026-09-05 to 2026-10-03
Pulled:  2026-10-12 01:36
Source: Real data from Open-Meteo historical archive API


**Markdown:** Here we printed where all the data came from, with as much detail as possible. It is important to cite information both to give credit to the source for using someone else's data, but also because crediting where you got it from means you can look back at the data later and confirm its accuracy.

### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

**Answer:** The chart would show a merged data set, just missing values because there would be no weather records that matched sale values. No one would catch it because without the error being raised, the data still runs, just produces information with missing values which would later lead to misleading analysis and conclusions. If get_weather returned an empty DataFrame on failure instead of raising, it would be impossible to tell if the empty response was from a failed call or from a really dry day. This means when you merge it into sales and chart rain against revenue, you might draw inconclusive patterns if you assume the missing value was not a fail (since you saw no error raised) .



### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [8]:
# stretch
charlottesville = get_weather('2026-09-05','2026-10-03',lat=38.03, lon=-78.51) #this is our original location we have been looking at
charlottesville = charlottesville[charlottesville['time'].isin(['2026-09-05','2026-10-03'])].copy() #making a copy so we don't manipulate the overarching data frame
charlottesville['city'] = 'Charlottesville' #adding a city column

orlando = get_weather('2026-09-05','2026-10-03',lat=28.54, lon=-81.38) #pulling data for orlando with the get_weather function
orlando = orlando[orlando['time'].isin(['2026-09-05','2026-10-03'])].copy() #making a copy so we don't alter the original data frame
orlando['city'] = 'Orlando' #adding a city column

both_cities = pd.concat([charlottesville,orlando],ignore_index = True) #concat so that we go one long-frame overall, and we ignore indexes because we don't care the index for each location, we want new indexes in the combined table
both_cities['date'] = pd.to_datetime(both_cities['time']) #makes sure that the date column is in the correct datetime format
both_cities = both_cities.rename(columns = {'temperature_2m_max':'temp_max','precipitation_sum':'precip_mm'}) #renames the columns as appropriate
both_cities = both_cities[['city','date','temp_max','precip_mm']] #specifies which columns we want included and in what order

print(both_cities)


              city       date  temp_max  precip_mm
0  Charlottesville 2026-09-05      29.4        2.5
1  Charlottesville 2026-10-03      22.0       14.7
2          Orlando 2026-09-05      32.1        3.9
3          Orlando 2026-10-03      30.4       14.5


**Markdown:** Here we pulled data from two cities and then added them to one table for comparison. This is important because in order to compare findings we don't want to have to search over numerous tables, but instead want the ability to look in one place to find everything we need. By putting the data into one long-format frame, we can see everything we need to just with this table, allowing for easier analysis and comparison.